# A Federated and Explainable Machine Learning Framework for Intrusion Detection in IIoT

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import (
    config, data_loader, preprocessing, balancing, feature_selection,
    optimization, models, evaluation, explainability, federated, visualization,
    export,
)

import random
random.seed(config.RANDOM_STATE)
np.random.seed(config.RANDOM_STATE)

print("Reproducible seed:", config.RANDOM_STATE)

## Phase 1 — Data Acquisition & Preprocessing

In [ ]:
raw = data_loader.load_raw_dataset()
print("Shape:", raw.shape)
raw_counts = raw[config.ATTACK_TYPE_COLUMN].value_counts().to_dict()
visualization.plot_class_distribution(raw_counts, "Class distribution — raw Edge-IIoTset")
plt.show()

In [ ]:
# Drop MITM / Fingerprinting (underrepresented), recode Ransomware -> Zero_Day,
# drop identifier/free-text columns, label-encode categoricals, z-score scale.
pre = preprocessing.preprocess(raw)
print("Features:", len(pre.feature_names), "| Classes:", pre.class_names)
print("\nPreprocessing report:")
for k in ("dropped_columns", "categorical_features", "numeric_features"):
    print(f"  {k}: {pre.report[k]}")

## Phase 2 — SMOTE Balancing & Random-Forest Feature Selection

In [ ]:
X_bal, y_bal = balancing.apply_smote(pre.X, pre.y)
after_counts = {pre.class_names[k]: int(v) for k, v in balancing.class_counts(y_bal).items()}
print("Balanced shape:", X_bal.shape)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
visualization.plot_class_distribution(
    {k: raw_counts.get(k, 0) for k in after_counts}, "Before SMOTE", ax=axes[0]
)
visualization.plot_class_distribution(after_counts, "After SMOTE (class parity)", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Random-Forest Gini importance ranking.
importance_df = feature_selection.compute_importances(X_bal, y_bal, pre.feature_names)
print("Top-20 discriminative features:")
display(importance_df.head(20))

## Phase 3 — Stratified Train/Test Split (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=config.TEST_SIZE, stratify=y_bal,
    random_state=config.RANDOM_STATE,
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## Phase 4 — Hyperparameter Optimization (Optuna / TPE)

The paper optimizes XGBoost and LightGBM with Optuna's TPE sampler, maximizing **weighted F1-score**. The final optimized values are in `src/config.py` (paper Table I). The cell below demonstrates the search on a sample; the tuned models are then built with the reported optima.

In [ ]:
# Optional: demonstrate a short TPE search on a 15% stratified sample.
demo = True
if demo:
    Xs, _, ys, _ = train_test_split(
        X_train, y_train, train_size=0.15, stratify=y_train,
        random_state=config.RANDOM_STATE,
    )
    study = optimization.run_study("xgboost", Xs, ys, n_trials=8)
    print("Optuna best weighted-F1 (demo):", round(study.best_value, 4))
    print("Best params:", study.best_params)
    print("\nReported optima used for the final model (Table I):")
    print("  XGBoost :", config.XGB_PARAMS)
    print("  LightGBM:", config.LGBM_PARAMS)

## Phase 5 — Model Training (baselines + hybrid ensemble + calibration)

In [ ]:
import time

trained = {}
train_times = {}

for name, build in [
    ("Random Forest", models.build_random_forest),
    ("CatBoost", models.build_catboost),
    ("XGBoost", models.build_xgboost),
    ("LGBM", models.build_lightgbm),
]:
    t0 = time.perf_counter()
    trained[name] = build().fit(X_train, y_train)
    train_times[name] = time.perf_counter() - t0
    print(f"{name}: trained in {train_times[name]:.1f}s")

In [ ]:
# Hybrid soft-voting ensemble: P(y|x) = 0.5 * P_xgb + 0.5 * P_lgbm (paper Eq. 5).
# Reuses the already-fitted base learners (no re-training).
hybrid = models.SoftVotingEnsemble(xgb=trained["XGBoost"], lgbm=trained["LGBM"])
hybrid.fit(X_train, y_train)
trained["Hybrid XGB-LGBM"] = hybrid
train_times["Hybrid XGB-LGBM"] = train_times["XGBoost"] + train_times["LGBM"]

# Isotonic probability calibration (paper Eq. 6, v=5).
t0 = time.perf_counter()
calibrated = models.calibrate(models.build_hybrid(), X_train, y_train, cv=config.CALIBRATION_CV)
train_times["Hybrid (calibrated)"] = time.perf_counter() - t0
print(f"Calibrated hybrid trained in {train_times['Hybrid (calibrated)']:.1f}s")

# Persist every model to outputs/models/ so a later session can reload without retraining.
from src import persistence
model_paths = persistence.save_all_models(trained, calibrated, config.MODELS_DIR)
print("Saved models:")
for name, path in model_paths.items():
    print(f"  {name}: {path}")

## Phase 6 — Evaluation & Benchmarking

In [ ]:
metrics_df = evaluation.build_metrics_table(trained, X_test, y_test)
metrics_df.loc["Hybrid XGB-LGBM (calibrated)"] = evaluation.evaluate_model(calibrated, X_test, y_test)
metrics_df = metrics_df[["accuracy", "balanced_accuracy", "precision", "recall", "f1"]]

print("Classification metrics (weighted) — test set:")
display(metrics_df.round(4))
print("\nPaper benchmark targets (Hybrid):", config.BENCHMARK_TARGETS)

In [ ]:
# Confusion matrices (normalized) for all models.
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, (name, model) in zip(axes.flat, list(trained.items()) + [("Hybrid (calibrated)", calibrated)]):
    cm = evaluation.build_confusion_matrix(y_test, model.predict(X_test))
    visualization.plot_confusion_matrix(cm, pre.class_names, name, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Computational efficiency (paper Table III): inference latency + memory footprint.
efficiency_rows = {}
for name, model in trained.items():
    row = evaluation.measure_efficiency(model, X_test)
    row["training_time_s"] = train_times.get(name, np.nan)
    efficiency_rows[name] = row
efficiency_df = pd.DataFrame(efficiency_rows).T
display(efficiency_df.round(4))

## Phase 7 — Explainable AI (SHAP)

In [ ]:
# Stratified sample for tractable SHAP computation.
shap_idx = explainability.shap_sample_indices(y_test, n_per_class=200)
X_shap = X_test[shap_idx]

shap_results = {}
for name in ("XGBoost", "LGBM"):
    values, _ = explainability.compute_shap_values(trained[name], X_shap)
    shap_results[name] = {
        "values": values,
        "importance": explainability.aggregate_feature_importance(values, pre.feature_names),
    }

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, name in zip(axes, ("XGBoost", "LGBM")):
    visualization.plot_shap_bar(
        shap_results[name]["importance"], config.SHAP_TOP_FEATURES,
        f"SHAP feature importance — {name}", ax=ax,
    )
plt.tight_layout()
plt.show()

In [ ]:
# SHAP summary (beeswarm) plots — predicted-class SHAP values.
import shap
for name in ("XGBoost", "LGBM"):
    print(f"\n{name} SHAP summary:")
    single = explainability.shap_values_predicted_class(
        trained[name], shap_results[name]["values"], X_shap
    )
    shap.summary_plot(single, X_shap, feature_names=pre.feature_names,
                      max_display=config.SHAP_TOP_FEATURES, show=False)
    plt.title(name)
    plt.show()

## Phase 8 — Federated Learning Simulation (non-IID + FedAvg)

In [ ]:
# Heterogeneous non-IID partition across N edge clients (Dirichlet).
client_indices = federated.dirichlet_partition(y_train, config.FL_N_CLIENTS, config.FL_NON_IID_ALPHA)
print("Client shard sizes:", [len(i) for i in client_indices])

client_models = federated.train_federated_clients(X_train, y_train, client_indices, learner="hybrid")

# Federated soft-voting aggregation (no raw data shared).
y_fed = federated.federated_predict(client_models, X_test)
fed_metrics = evaluation.compute_metrics(y_test, y_fed)
print("\nFederated global model (test):")
for k, v in fed_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Federated consistency verification: feature-importance agreement across nodes.
client_importances = federated.client_feature_importances(X_train, y_train, client_indices, pre.feature_names)
visualization.plot_federated_importance(client_importances, pre.feature_names, top_k=config.SHAP_TOP_FEATURES)
plt.show()

## Phase 9 — Production Model Export

Export portable, production-grade artifacts (native XGBoost/LightGBM/CatBoost formats, ONNX, ensemble spec, and a metadata manifest with integrity hashes) to `outputs/export/`.

In [ ]:
# Portable artifacts: native formats + ONNX + ensemble spec + metadata manifest.
# ONNX is optional and degrades gracefully if its deps are not installed.
export_artifacts = export.export_all(
    trained, metrics_df=metrics_df,
    class_names=pre.class_names, feature_names=pre.feature_names,
    onnx=True,
)
print("Exported artifacts:")
for name, path in export_artifacts.items():
    print(f"  {name}: {path}")


## Summary

The hybrid **XGB-LGBM** soft-voting ensemble (with isotonic calibration) is the top performer across accuracy, balanced accuracy, precision, recall, and F1-score, while SHAP exposes the key protocol features driving detections and the federated simulation preserves feature-importance consistency across decentralized IIoT nodes.

All artifacts (tables/figures) are also exported to `outputs/`. To extend the work, add new models in `src/models.py`, new FL aggregation strategies in `src/federated.py`, or new explainers in `src/explainability.py` — the orchestration in `src/pipeline.py` and this notebook stays unchanged.